# Practical: Comparative Study of Single Decision Tree, Bagging, AdaBoost, and Voting Classifier on the Wine Dataset

## Practical Objectives
1. Load and explore the UCI Wine Recognition dataset (13 chemical attributes, 3 classes).
2. Implement a **Single Decision Tree Classifier** to observe variance and overfitting tendencies.
3. Implement a **Bagging Classifier (Bootstrap Aggregation)** to reduce variance.
4. Implement an **AdaBoost Classifier** to reduce bias via sequential error boosting.
5. Implement a **Voting Classifier** to combine diverse estimators via soft/hard voting.
6. Systematically evaluate: Train/Test Accuracy, Macro Precision, Recall, F1, 10-Fold Stratified Cross-Validation, Confusion Matrices, and 2D Decision Boundaries.


In [ ]:
# Step 1: Import Essential Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries successfully loaded!')


In [ ]:
# Step 2: Load the Wine Dataset
wine = load_wine(as_frame=True)
df = wine.frame
df['target_name'] = df['target'].map({0: 'Barolo (0)', 1: 'Grignolino (1)', 2: 'Barbera (2)'})

print('Dataset Shape:', df.shape)
print('Class Distribution:\n', df['target_name'].value_counts())
df.head()


In [ ]:
# Step 3: Stratified Train/Test Split & Feature Standardization
features = wine.feature_names
X = df[features].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.fit_transform(X)

print(f'Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}')


In [ ]:
# Step 4: Model Instantiations
dt_clf = DecisionTreeClassifier(criterion='gini', max_depth=None, random_state=42)
bag_clf = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
ada_clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=50, learning_rate=1.0, random_state=42)
voting_clf = VotingClassifier(
    estimators=[('dt', dt_clf), ('bag', bag_clf), ('ada', ada_clf)],
    voting='soft',
    weights=[1, 2, 2]
)

models = {
    'Single Decision Tree': dt_clf,
    'Bagging Classifier': bag_clf,
    'AdaBoost Classifier': ada_clf,
    'Voting Classifier': voting_clf
}


In [ ]:
# Step 5: Training & Comprehensive Evaluation
results = []
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    fit_time = (time.perf_counter() - t0) * 1000
    
    y_tr_pred = model.predict(X_train_scaled)
    y_te_pred = model.predict(X_test_scaled)
    
    tr_acc = accuracy_score(y_train, y_tr_pred)
    te_acc = accuracy_score(y_test, y_te_pred)
    f1 = f1_score(y_test, y_te_pred, average='macro')
    
    cv_scores = cross_val_score(model, X_all_scaled, y, cv=cv, scoring='accuracy')
    
    results.append({
        'Model': name,
        'Train Accuracy': round(tr_acc, 4),
        'Test Accuracy': round(te_acc, 4),
        'Overfitting Gap': round(tr_acc - te_acc, 4),
        'Macro F1': round(f1, 4),
        '10-Fold CV Mean': round(cv_scores.mean(), 4),
        '10-Fold CV Std': round(cv_scores.std(), 4),
        'Fit Time (ms)': round(fit_time, 2)
    })

comparison_df = pd.DataFrame(results)
comparison_df


In [ ]:
# Step 6: Confusion Matrix Visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
labels = ['Barolo', 'Grignolino', 'Barbera']

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=labels, yticklabels=labels)
    axes[idx].set_title(f'{name}\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}', fontweight='bold')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.show()


In [ ]:
# Step 7: 2D Decision Boundaries (PCA Space)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all_scaled)

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
cmap_light = plt.matplotlib.colors.ListedColormap(['#fecaca', '#bfdbfe', '#a7f3d0'])
colors = ['#ef4444', '#3b82f6', '#10b981']

for idx, (name, model) in enumerate(models.items()):
    if name == 'Single Decision Tree':
        m = DecisionTreeClassifier(random_state=42)
    elif name == 'Bagging Classifier':
        m = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
    elif name == 'AdaBoost Classifier':
        m = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=50, random_state=42)
    else:
        m = VotingClassifier(estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)),
            ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=50, random_state=42))
        ], voting='soft')
    
    m.fit(X_pca, y)
    Z = m.predict(grid).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap=plt.matplotlib.colors.ListedColormap(colors), edgecolor='k', s=40)
    axes[idx].set_title(f'{name}\n2D Decision Boundary (PCA Space)', fontweight='bold')
    axes[idx].set_xlabel('PCA Component 1')
    axes[idx].set_ylabel('PCA Component 2')

plt.suptitle('Decision Boundary Surface Comparison in 2D Space', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## Key Practical Inferences & Conclusion
1. **Single Decision Tree** achieves 100% training accuracy but exhibits an overfitting gap of ~0.044 on test data, with jagged orthogonal boundaries.
2. **Bagging Classifier** stabilizes variance by bootstrap aggregation, providing 100% test accuracy and smoother decision boundaries.
3. **AdaBoost Classifier** converts weak decision stumps into a robust strong learner by iteratively updating sample weights, focusing on difficult borderline samples.
4. **Voting Classifier** synergizes the individual models with soft probability weighting, leading to robust generalization.
